# Patch2Prod Small SFT Starter

This notebook is a small, credit-conscious starting point for training on Patch2Prod Arena.

It does four things:
- bootstraps a tiny SFT dataset if the repo dataset is too small,
- fine-tunes a small instruct model with LoRA,
- saves a loss curve image for the hackathon submission,
- runs a quick generation sanity check.

Recommended first run:
- base model: `Qwen/Qwen2.5-0.5B-Instruct`
- epochs: `3`
- LoRA only
- small batch size

After this works, we can add a GRPO notebook.

In [ ]:
!pip install -q -r training/requirements-train.txt peft bitsandbytes sentencepiece

In [ ]:
from pathlib import Path
import json
import os
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "patch2prod").exists():
    raise RuntimeError("Run this notebook from the repo root so relative paths resolve correctly.")

sys.path.insert(0, str(REPO_ROOT))

random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print({"device": device, "cuda": torch.cuda.is_available()})

In [ ]:
BASE_MODEL = os.getenv("BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
OUTPUT_DIR = REPO_ROOT / "outputs" / "sft_patch2prod_small"
PLOT_DIR = REPO_ROOT / "artifacts" / "plots"
BOOTSTRAP_DATA_PATH = REPO_ROOT / "data" / "sft_traces_bootstrap.jsonl"
MAX_STEPS_PER_SAMPLE = 8
NUM_EPOCHS = 3
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "base_model": BASE_MODEL,
    "output_dir": str(OUTPUT_DIR),
    "bootstrap_data": str(BOOTSTRAP_DATA_PATH),
})

In [ ]:
def json_action_completion(actions):
    return json.dumps({"actions": actions}, ensure_ascii=True)


seed_rows = [
    {
        "prompt": "Task authsdk_mobile_contract_break: auth-service CI failed after authsdk upgrade. Return JSON actions only.",
        "completion": json_action_completion([
            {"action": "view_log", "job_name": "unit-tests"},
            {"action": "view_commit_history"},
            {"action": "view_diff", "commit_id": "c42"},
            {"action": "submit_causal_change", "commit": "c42", "summary": "authsdk upgraded from 1.8.0 to 2.0.0"},
            {"action": "view_migration_guide"},
            {"action": "replace", "file_path": "app/retry.py", "search": "build_retry_policy", "replace": "create_retry_policy"},
            {"action": "run_unit_tests", "service": "auth-service"},
            {"action": "submit_release_decision", "decision": "block", "reason": "Local CI is repaired but mobile-gateway can still break from token_expiry contract drift", "owner_to_notify": "mobile-platform"}
        ])
    },
    {
        "prompt": "Task authsdk_mobile_contract_break: auth-service CI failed after authsdk upgrade. Be risk-aware and check downstream impact. Return JSON actions only.",
        "completion": json_action_completion([
            {"action": "view_log", "job_name": "unit-tests"},
            {"action": "view_diff", "commit_id": "c42"},
            {"action": "cat", "file_path": "app/retry.py"},
            {"action": "view_migration_guide"},
            {"action": "replace", "file_path": "app/retry.py", "search": "build_retry_policy", "replace": "create_retry_policy"},
            {"action": "run_unit_tests", "service": "auth-service"},
            {"action": "view_dependency_graph", "service": "auth-service"},
            {"action": "submit_release_decision", "decision": "block", "reason": "Downstream services depend on auth token behavior and require targeted contract validation", "owner_to_notify": "mobile-platform"}
        ])
    },
    {
        "prompt": "Task payment_schema_checkout_break: payment-service tests fail after schema rename. Return JSON actions only.",
        "completion": json_action_completion([
            {"action": "view_log", "job_name": "integration-tests"},
            {"action": "view_commit_history"},
            {"action": "view_diff", "commit_id": "p17"},
            {"action": "submit_causal_change", "commit": "p17", "summary": "payment_status renamed to status"},
            {"action": "cat", "file_path": "app/payment_response.py"},
            {"action": "view_migration_guide"},
            {"action": "replace", "file_path": "app/payment_response.py", "search": "return {'status': p.status, 'id': p.id}", "replace": "return {'status': p.status, 'payment_status': p.status, 'id': p.id}"},
            {"action": "submit_release_decision", "decision": "ship", "reason": "The backward-compatible payload restores checkout expectations", "owner_to_notify": "checkout-platform"}
        ])
    },
    {
        "prompt": "Task payment_schema_checkout_break: payment-service changed a response field and checkout still expects the old contract. Return JSON actions only.",
        "completion": json_action_completion([
            {"action": "view_log", "job_name": "integration-tests"},
            {"action": "view_diff", "commit_id": "p17"},
            {"action": "view_migration_guide"},
            {"action": "replace", "file_path": "app/payment_response.py", "search": "return {'status': p.status, 'id': p.id}", "replace": "return {'status': p.status, 'payment_status': p.status, 'id': p.id}"},
            {"action": "run_unit_tests", "service": "payment-service"},
            {"action": "view_dependency_graph", "service": "payment-service"},
            {"action": "run_contract_tests", "service": "checkout-service"},
            {"action": "submit_release_decision", "decision": "ship", "reason": "Local CI and downstream checkout contract both pass after backward-compatible response fix", "owner_to_notify": "checkout-platform"}
        ])
    }
]

with BOOTSTRAP_DATA_PATH.open("w", encoding="utf-8") as f:
    for row in seed_rows:
        f.write(json.dumps(row, ensure_ascii=True) + "\n")

print(f"wrote {len(seed_rows)} bootstrap SFT rows to {BOOTSTRAP_DATA_PATH}")

In [ ]:
def load_sft_jsonl(path: Path) -> Dataset:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            rows.append({"text": row["prompt"] + "\n" + row["completion"]})
    return Dataset.from_list(rows)


dataset = load_sft_jsonl(BOOTSTRAP_DATA_PATH)
print(dataset)
pd.DataFrame(dataset[:])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=1,
    save_strategy="epoch",
    report_to=[],
    max_seq_length=768,
    bf16=torch.cuda.is_available(),
    fp16=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=config,
)

train_result = trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
train_result

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
loss_history = history.dropna(subset=["loss"])[["step", "loss"]].copy()
loss_history

In [ ]:
loss_curve_path = PLOT_DIR / "loss_curve.png"

plt.figure(figsize=(7, 4))
plt.plot(loss_history["step"], loss_history["loss"], marker="o")
plt.title("Patch2Prod Small SFT Loss Curve")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.tight_layout()
plt.savefig(loss_curve_path)
plt.show()

print(f"saved loss curve to {loss_curve_path}")

In [ ]:
test_prompt = """You are a release engineering agent in Patch2Prod Arena.

Goal:
Fix the failing CI, identify the causal change, reason about downstream blast radius,
run targeted validation, and make the correct release decision.

Task ID: authsdk_mobile_contract_break
Service: auth-service
Failure: ImportError: cannot import name 'build_retry_policy' from authsdk.helpers

Return ONLY valid JSON:
{"actions":[{"action":"..."}]}
"""

inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(generated)

## Next Steps

After this notebook works end to end, the next improvements should be:

- expand `data/sft_traces.jsonl` with more diverse successful traces,
- evaluate the small checkpoint with `training/evaluate.py`,
- create a second notebook for GRPO / RL with explicit anti-cheat rewards,
- inspect generations manually for reward hacking or brittle shortcuts.
